In [5]:
import sys
import os
import random
import pandas as pd

from pathlib import Path
import sys

try:
    ROOT = Path(__file__).resolve().parents[1]
except NameError:
    ROOT = Path.cwd().parent

sys.path.append(str(ROOT / "src"))

RESULTS = ROOT / "experiments" / "technology diffusion"
RESULTS.mkdir(parents=True, exist_ok=True)

from technology_diffusion import *

In [6]:
c_list = [1, 5, 10, 20]
n_list = [200, 400, 600, 1000, 2000]
seed_list = [1, 42, 53, 99, 101]

In [12]:
init_nodes = 5
init_mode = "complete"

delta = 0.5
xi = 0.1
d = 2
buffer_dim = 5000
verbose = 0

strategy = [degree_threshold, high_thetas_start, SD_start, degree_connected, degree, betweenness, random_start]

save_results_csv = True
results_csv_path = RESULTS / "technology_diffusion_results.csv"
save_gurobi_log = True
gurobi_log_path = RESULTS / "gurobi.log"

combinations = list(itertools.product(n_list, c_list, seed_list))
print(f"Total runs: {len(combinations)}")

results = []
start_all = time.time()

for run_idx, combo in enumerate(combinations, start=1):
    n_nodes, c, seed = combo
    max_time = 180 * (n_nodes//100)
    max_time = 0.01

    print("\n" + "=" * 90)
    print(
        f"Run {run_idx}/{len(combinations)} | "
        f"N={n_nodes}, c={c}, seed={seed}, init_nodes={init_nodes}, init_mode={init_mode}"
    )

    g, thetas = create_pa_graph(
        n_nodes=n_nodes,
        c=c,
        seed=seed,
        init_nodes=init_nodes,
        init_mode=init_mode,
    )

    # Goldberg-Liu IP
    print(f"Building Goldberg-Liu IP...", end='')
    start = time.time()
    model, x = build_golberg_liu_ip(g, thetas, max_time)
    build_time = time.time() - start

    if max_time - build_time < 0:
        print(f'\rGoldberg-Liu IP build failed: build time {round(build_time, 2)}s exceeds max time {max_time}s.' + ' ' * 20)
        gl_build_failed = True
        gl_build_time = None
        gl_opt_time = None
        gl_runtime = None
        gl_k = None

    else:
        print(f'\rGoldberg-Liu IP built in {round(build_time, 2)} seconds!' + ' ' * 20)
        print(f"Solving Goldberg-Liu IP with time limit {round(max_time-build_time, 2)} seconds...", end='')
        suppress_print()
        model.setParam("OutputFlag", 1 if save_gurobi_log else 0)
        model.setParam("LogToConsole", 0)
        model.setParam("LogFile", str(gurobi_log_path) if save_gurobi_log else "")
        model.setParam("TimeLimit", max(max_time - build_time, 60))
        resume_print()
        model.optimize()
        print(f'\rGoldberg-Liu IP solved in {round(model.Runtime, 2)} seconds!' + ' ' * 20)

        gl_build_failed = False
        gl_build_time = round(build_time, 4)
        gl_opt_time = round(float(model.Runtime), 4)
        gl_runtime = round(build_time + gl_opt_time, 4)
        gl_k = int(round(model.objVal)) if model.SolCount > 0 else None

    # NS binary search
    ns_k, final_x, ns_runtime = NS_technology_diffusion_binary_search(
        g,
        thetas,
        strategy,
        delta,
        xi,
        d,
        max_time,
        buffer_dim,
        verbose=verbose,
    )

    gap = (ns_k - gl_k) if (ns_k is not None and gl_k is not None) else None

    results.append([
        n_nodes,
        c,
        seed,
        g.number_of_nodes(),
        g.number_of_edges(),
        max_time,
        gl_build_failed,
        gl_build_time,
        gl_opt_time,
        gl_runtime,
        gl_k,
        ns_runtime,
        ns_k,
        gap,
    ])

    if gl_build_failed:
        print(f"Goldberg-Liu -> Build failed.")
    else:
        print(f"Goldberg-Liu -> K={gl_k}, runtime={round(gl_runtime, 2)}s")
    print(f"NS Binary    -> K={ns_k}, runtime={round(float(ns_runtime), 2)}s, " f"gap(NS-GL)={gap}")

elapsed_all = time.time() - start_all

columns = [
    "n_nodes",
    "c",
    "seed",
    "num_nodes",
    "num_edges",
    "max_time",
    "GL_build_failed",
    "GL_build_time_s",
    "GL_optimization_time_s",
    "GL_runtime_s",
    "GL_K",
    "NS_runtime_s",
    "NS_K",
    "NS_GL_gap",
]

results_df = pd.DataFrame(results, columns=columns)
results_df = results_df.sort_values(["n_nodes", "c", "seed"]).reset_index(drop=True)

print("\n" + "=" * 90)
print(f"Completed {len(results_df)} runs in {round(elapsed_all, 2)} seconds.")
display(results_df)

if save_results_csv:
    results_df.to_csv(results_csv_path, index=False)
    print(f"Results saved to: {results_csv_path}")

if save_gurobi_log:
    print(f"Gurobi log saved to: {gurobi_log_path}")

Total runs: 100

Run 1/100 | N=200, c=1, seed=1, init_nodes=5, init_mode=complete
Goldberg-Liu IP build failed: build time 0.05s exceeds max time 0.01s.                    
Goldberg-Liu -> Build failed.
NS Binary    -> K=25, runtime=0.02s, gap(NS-GL)=None

Run 2/100 | N=200, c=1, seed=42, init_nodes=5, init_mode=complete
Goldberg-Liu IP build failed: build time 0.05s exceeds max time 0.01s.                    
Goldberg-Liu -> Build failed.
NS Binary    -> K=18, runtime=0.01s, gap(NS-GL)=None

Run 3/100 | N=200, c=1, seed=53, init_nodes=5, init_mode=complete
Goldberg-Liu IP build failed: build time 0.05s exceeds max time 0.01s.                    
Goldberg-Liu -> Build failed.
NS Binary    -> K=25, runtime=0.01s, gap(NS-GL)=None

Run 4/100 | N=200, c=1, seed=99, init_nodes=5, init_mode=complete
Goldberg-Liu IP build failed: build time 0.05s exceeds max time 0.01s.                    
Goldberg-Liu -> Build failed.
NS Binary    -> K=25, runtime=0.01s, gap(NS-GL)=None

Run 5/100 | N=200, c

,n_nodes,c,seed,num_nodes,num_edges,max_time,GL_build_failed,GL_build_time_s,GL_optimization_time_s,GL_runtime_s,GL_K,NS_runtime_s,NS_K,NS_GL_gap
0,200,1,1,200,506,0.01,True,None,None,None,None,0.017041,25,None
1,200,1,42,200,514,0.01,True,None,None,None,None,0.012079,18,None
2,200,1,53,200,488,0.01,True,None,None,None,None,0.011052,25,None
3,200,1,99,200,502,0.01,True,None,None,None,None,0.013218,25,None
4,200,1,101,200,479,0.01,True,None,None,None,None,0.012537,55,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2000,20,1,2000,5038,0.01,True,None,None,None,None,0.014028,1000,None
96,2000,20,42,2000,4971,0.01,True,None,None,None,None,0.014600,1000,None
97,2000,20,53,2000,4960,0.01,True,None,None,None,None,28.989537,1003,None
98,2000,20,99,2000,4937,0.01,True,None,None,None,None,27.330569,1006,None


Results saved to: /Users/matteobergamaschi/Desktop/td_git/experiments/technology diffusion/technology_diffusion_results.csv
Gurobi log saved to: /Users/matteobergamaschi/Desktop/td_git/experiments/technology diffusion/gurobi.log


In [ ]:
g, thetas = create_pa_graph(2000, 20, seed=101, init_nodes=5, init_mode="complete")
K = 1000
max_time = 0.1
for _ in range(50):
    x0 = random_start(g, n_nodes, K, thetas, connected = 1)
    NS_matteo(g, thetas, x0, delta, xi, d, max_time, buffer_dim, verbose=0, min_conn=20, mg_max_depth=10)

In [19]:
for _ in range(5):
    x0 = random_start(g, n_nodes, K, thetas, connected = 1)
    Neighbor_Search(g, thetas, x0, delta, xi, d, max_time, buffer_dim, verbose = verbose)

KeyboardInterrupt: 